In [ ]:
# ============================================================
# DAY 22: CUSTOM CNN - RESULTS + GPU TRAINING
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)


# ------------------------------------------------------------
# 2. DEVICE CHECK
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available - using CPU")


# ------------------------------------------------------------
# 3. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_ROOT)


# ------------------------------------------------------------
# 4. CUSTOM CNN MODEL
# ------------------------------------------------------------

class CustomCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 28 * 28,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(128, 1)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# 5. CREATE MODEL
# ------------------------------------------------------------

model = CustomCNN().to(device)

print(model)


# ------------------------------------------------------------
# 6. LOSS FUNCTION + OPTIMIZER
# ------------------------------------------------------------

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 7. TRAINING FUNCTION
# ------------------------------------------------------------

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)

        labels = (
            labels.float()
            .to(device)
            .unsqueeze(1)
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        running_loss / len(loader),
        correct / total
    )


# ------------------------------------------------------------
# 8. VALIDATION FUNCTION
# ------------------------------------------------------------

def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = (
                labels.float()
                .to(device)
                .unsqueeze(1)
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            predictions = (
                torch.sigmoid(outputs) >= 0.5
            ).float()

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        running_loss / len(loader),
        correct / total
    )


# ------------------------------------------------------------
# 9. TRAINING SETTINGS
# ------------------------------------------------------------

EPOCHS = 5

train_losses = []
val_losses = []

train_accs = []
val_accs = []


# ------------------------------------------------------------
# 10. FULL DATASET TRAINING
# ------------------------------------------------------------

for epoch in range(EPOCHS):

    # Validate that the data loaders were created in a prior cell.
    missing_loaders = [
        name for name in ("train_loader", "val_loader", "test_loader")
        if name not in globals()
    ]

    if missing_loaders:
        raise NameError(
            "Missing required loader(s): "
            + ", ".join(missing_loaders)
            + ". Run the data preparation / loader-creation cell before training."
        )

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


# ------------------------------------------------------------
# 11. SAVE TRAINED MODEL
# ------------------------------------------------------------

model_path = MODEL_DIR / "custom_cnn_full.pth"

torch.save(
    model.state_dict(),
    model_path
)

print("\nModel saved at:")
print(model_path)


# ------------------------------------------------------------
# 12. TEST SET EVALUATION
# ------------------------------------------------------------

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).int()

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu()
            .numpy()
            .flatten()
        )

        all_probabilities.extend(
            probabilities.cpu()
            .numpy()
            .flatten()
        )


# ------------------------------------------------------------
# 13. CALCULATE FINAL METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

roc_auc = roc_auc_score(
    all_labels,
    all_probabilities
)

cm = confusion_matrix(
    all_labels,
    all_predictions
)


# ------------------------------------------------------------
# 14. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nCustom CNN - Full Training Results")
print("-----------------------------------")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "Model": ["Custom CNN"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "ROC_AUC": [roc_auc]
})

results_path = RESULTS_DIR / "custom_cnn_results.csv"

results.to_csv(
    results_path,
    index=False
)

print("\nResults saved at:")
print(results_path)

Device: cpu
GPU not available - using CPU
Project: c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION
CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


NameError: name 'train_loader' is not defined